# 01: Pull Stadium & Game Data

Pulls raw, unprocessed data for the NWSL infrastructure analysis from two sources:

1. **American Soccer Analysis (ASA) API**: games, players, teams, stadia, and star-player team/season rosters for NWSL and MLS (`itscalledsoccer` package).
2. **Wikipedia**: supplementary stadium metadata (via `pd.read_html` table scraping), used later to fill in details the ASA API doesn't provide.

This notebook does **no filtering, deduplication, or merging**: that happens in `02_stadiums_clean.ipynb`. Its only job is to fetch data and save it to `data/raw/`.

**Inputs:** none (pulls live from ASA API + Wikipedia)
**Outputs:** `data/raw/games_raw.csv`, `data/raw/players_raw.csv`, `data/raw/teams_raw.csv`, `data/raw/stadia_raw.csv`, `data/raw/stadiums_wiki_nwsl_raw.csv`, `data/raw/stadiums_wiki_mls_raw.csv`, `data/raw/star_player_seasons_raw.csv`

In [ ]:
from itscalledsoccer import AmericanSoccerAnalysis
import pandas as pd
import requests
import os

# Initialize client (no authentication required)
asa = AmericanSoccerAnalysis()

# data/ lives one level up from code/
DATA_RAW_DIR = os.path.join("..", "data", "raw")
os.makedirs(DATA_RAW_DIR, exist_ok=True)


## Games, players, teams, stadia (ASA API)

In [ ]:
# Grab all NWSL and MLS games
nwsl_games = asa.get_games(leagues=["nwsl"])
mls_games = asa.get_games(leagues=["mls"])
nwsl_games["league"] = "nwsl"
mls_games["league"] = "mls"
games = pd.concat([nwsl_games, mls_games])

# Diagnostic: check for missing values before saving
print("Missing value % by column:")
print(games.isna().mean() * 100)
# Note: ~30% missing stadium_id- will check more closely in the cleaning notebook

games.to_csv(os.path.join(DATA_RAW_DIR, "games_raw.csv"), index=False)
print(f"Saved {len(games)} rows to games_raw.csv")


In [ ]:
# Same interface for all entity types
nwsl_players = asa.get_players(leagues="nwsl")
mls_players = asa.get_players(leagues="mls")
nwsl_players = nwsl_players.assign(league="nwsl")
mls_players = mls_players.assign(league="mls")
players = pd.concat([nwsl_players, mls_players])

nwsl_teams = asa.get_teams(leagues="nwsl")
mls_teams = asa.get_teams(leagues="mls")
nwsl_teams = nwsl_teams.assign(league="nwsl")
mls_teams = mls_teams.assign(league="mls")
teams = pd.concat([nwsl_teams, mls_teams])

stadia = asa.get_stadia(leagues=["nwsl", "mls"])

print(f"players: {len(players)} rows, teams: {len(teams)} rows, stadia: {len(stadia)} rows")

players.to_csv(os.path.join(DATA_RAW_DIR, "players_raw.csv"), index=False)
teams.to_csv(os.path.join(DATA_RAW_DIR, "teams_raw.csv"), index=False)
stadia.to_csv(os.path.join(DATA_RAW_DIR, "stadia_raw.csv"), index=False)
print("Saved players_raw.csv, teams_raw.csv, stadia_raw.csv")


## Star player rosters, by team and season

For the "does a star player move the needle on attendance?" question in `03_stadiums_analyze.ipynb`. Limited to the three players we already have Google Trends search-interest data for (`time_series_nwsl_stars.csv`) (Megan Rapinoe, Alex Morgan, Trinity Rodman) rather than pulling full league-wide rosters, since those are the only three we can cross-reference against public interest.

`get_player_xgoals(..., split_by_teams=True, split_by_seasons=True)` returns one row per player/team/season with minutes played, which is enough to tell which team a player was on, and how big a role they played, each season (not full per-game lineups, but sufficient roster-level granularity).

In [ ]:
STAR_PLAYER_NAMES = ["Megan Rapinoe", "Alex Morgan", "Trinity Rodman"]

# Look up player_ids by exact name match against the NWSL player table pulled above,
# rather than hardcoding opaque ASA player_id strings
nwsl_players_only = players[players["league"] == "nwsl"]
star_ids = nwsl_players_only.loc[
    nwsl_players_only["player_name"].isin(STAR_PLAYER_NAMES), "player_id"
].tolist()

assert len(star_ids) == len(STAR_PLAYER_NAMES), (
    f"Expected {len(STAR_PLAYER_NAMES)} player_ids, found {len(star_ids)} -- "
    "an exact name match may have failed (e.g. a name change in the API)."
)

star_player_seasons = asa.get_player_xgoals(
    leagues=["nwsl"], player_ids=star_ids, split_by_teams=True, split_by_seasons=True,
)

star_player_seasons.to_csv(os.path.join(DATA_RAW_DIR, "star_player_seasons_raw.csv"), index=False)
print(f"Saved {len(star_player_seasons)} rows to star_player_seasons_raw.csv")
star_player_seasons[["player_id", "team_id", "season_name", "minutes_played"]]


## Supplementary stadium metadata (Wikipedia scrape)

The ASA API's stadium table has a lot of missing metadata (capacity, location, etc.), so also scraping the "List of NWSL/MLS stadiums" Wikipedia pages for a second source to merge in during cleaning.


In [ ]:
url_nwsl = "https://en.wikipedia.org/wiki/List_of_National_Women%27s_Soccer_League_stadiums"
url_mls = "https://en.wikipedia.org/wiki/List_of_Major_League_Soccer_stadiums"

# Avoid the 403 Forbidden error Wikipedia returns for default request headers
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response_nwsl = requests.get(url_nwsl, headers=headers)
response_mls = requests.get(url_mls, headers=headers)

tables_nwsl = pd.read_html(response_nwsl.text)
tables_mls = pd.read_html(response_mls.text)

# The "Primary stadiums" table is table index 0 for the NWSL page, index 1 for the MLS page
stadiums_wiki_nwsl = tables_nwsl[0]
stadiums_wiki_mls = tables_mls[1]

print(f"NWSL wiki table: {stadiums_wiki_nwsl.shape}, MLS wiki table: {stadiums_wiki_mls.shape}")

stadiums_wiki_nwsl.to_csv(os.path.join(DATA_RAW_DIR, "stadiums_wiki_nwsl_raw.csv"), index=False)
stadiums_wiki_mls.to_csv(os.path.join(DATA_RAW_DIR, "stadiums_wiki_mls_raw.csv"), index=False)
print("Saved stadiums_wiki_nwsl_raw.csv, stadiums_wiki_mls_raw.csv")
